# 02 — Dataset Creation (Supervised & Unsupervised)

**Thesis:** Structured Subsystem-Aware Feature Representations for Predictive Maintenance in Metro Systems  
**Author:** Emirhan Kurtulus, TU Wien

Creates the windowed feature datasets matching **Steiner's baseline pipeline** (RAMS 2026)  
for a controlled comparison in Phase 2 (RQ2).

**Baseline configs (best from Steiner's ablation):**

| Model | Window | Label Horizon | Dev F1 | Test F1 |
|-------|--------|--------------|--------|--------|
| RF (supervised) | 350s (6 min) | 22326s (6.2h) | 0.98 | 0.21 |
| LSTM-AE (unsupervised) | 1402s (23 min) | 11163s (3.1h) | 0.75 | 0.08 |

**Outputs:**
- `outputs/supervised/train.parquet`, `test.parquet`
- `outputs/unsupervised/train.npy`, `val.npy`, `dev_eval.npy`, `test.npy`, `metadata.json`

## 0. Setup

In [1]:
import sys, os, gc, json
import pandas as pd
import numpy as np
import glob
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.abspath('..'))
from src.config import SUBSYSTEMS

ROOT = os.path.abspath('..')
TRAIN_GLOB = os.path.join(ROOT, 'train', '**', '*.parquet')
TEST_GLOB  = os.path.join(ROOT, 'test',  '**', '*.parquet')

# ── Sensor classification (matching Steiner's 96 sensors) ──
# Steiner excludes: TRAIN_LINE, TRAIN_CURRENT_SECTION, TRAIN_IS_SPECIAL_SECTION,
#   TRAIN_MANUAL_MODE, TRAIN_AUTOMATIC_MODE, TRAIN_EMERGENCY_MODE
EXCLUDED_CONTEXT = {
    'TRAIN_LINE', 'TRAIN_CURRENT_SECTION', 'TRAIN_IS_SPECIAL_SECTION',
    'TRAIN_MANUAL_MODE', 'TRAIN_AUTOMATIC_MODE', 'TRAIN_EMERGENCY_MODE',
}

BINARY_SENSORS = [
    # Spring brake active (CW only)
    'CW1_SPRING_BRAKE_ACTIVE_BOGIE1', 'CW1_SPRING_BRAKE_ACTIVE_BOGIE2',
    'CW2_SPRING_BRAKE_ACTIVE_BOGIE1', 'CW2_SPRING_BRAKE_ACTIVE_BOGIE2',
    # Proportional valve pressure available (all wagons)
    'CW1_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE1', 'CW1_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE2',
    'CW2_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE1', 'CW2_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE2',
    'MW1_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE1', 'MW1_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE2',
    'MW2_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE1', 'MW2_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE2',
    'MW3_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE1', 'MW3_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE2',
    'MW4_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE1', 'MW4_PROPORTIONAL_VALVE_PRESSURE_AVAILABLE_BOGIE2',
    # Pneumatic brake active (CW only)
    'CW1_PNEUMATIC_BRAKE_ACTIVE', 'CW2_PNEUMATIC_BRAKE_ACTIVE',
    # Compressor running
    'CW1_COMPRESSOR_RUNNING', 'CW2_COMPRESSOR_RUNNING',
    # Brake signal
    'TRAIN_BRAKE_SIGNAL',
]

ANALOG_SENSORS = [
    c for ss, cols in SUBSYSTEMS.items() for c in cols
    if c not in BINARY_SENSORS and c not in EXCLUDED_CONTEXT
]

ALL_SENSORS = ANALOG_SENSORS + BINARY_SENSORS

# ── Baseline hyperparameters (from Steiner RAMS 2026) ──
# Supervised (RF)
SUP_WINDOW_S  = 350    # 6 min feature aggregation window
SUP_HORIZON_S = 22326  # 6.2h label shift

# Unsupervised (LSTM-AE)
UNSUP_SEQ_S    = 1402   # 23 min sequence window
UNSUP_HORIZON_S = 11163  # 3.1h label shift

# Meta columns
TS_COL   = 'TIMESTAMP'
FAIL_COL = 'TRAIN_IS_IN_FAILURE'
MAINT_COL = 'TRAIN_IS_IN_MAINTENANCE'

# Output paths
SUP_OUT   = os.path.join(ROOT, 'outputs', 'supervised')
UNSUP_OUT = os.path.join(ROOT, 'outputs', 'unsupervised')
os.makedirs(SUP_OUT, exist_ok=True)
os.makedirs(UNSUP_OUT, exist_ok=True)

print(f'Analog sensors:  {len(ANALOG_SENSORS)}')
print(f'Binary sensors:  {len(BINARY_SENSORS)}')
print(f'Total sensors:   {len(ALL_SENSORS)}')
print(f'\nSupervised:   Wfeat={SUP_WINDOW_S}s ({SUP_WINDOW_S/60:.0f}min), Wlabel={SUP_HORIZON_S}s ({SUP_HORIZON_S/3600:.1f}h)')
print(f'Unsupervised: Wseq={UNSUP_SEQ_S}s ({UNSUP_SEQ_S/60:.0f}min), Wlabel={UNSUP_HORIZON_S}s ({UNSUP_HORIZON_S/3600:.1f}h)')
print('Setup done.')

Analog sensors:  74
Binary sensors:  21
Total sensors:   95

Supervised:   Wfeat=350s (6min), Wlabel=22326s (6.2h)
Unsupervised: Wseq=1402s (23min), Wlabel=11163s (3.1h)
Setup done.


## 1. Helper Functions

In [2]:
def load_split(glob_pattern, label):
    """Load all parquet files, sort by timestamp."""
    files = sorted(glob.glob(glob_pattern, recursive=True))
    print(f'[{label}] Loading {len(files)} files...')
    df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
    df[TS_COL] = pd.to_datetime(df[TS_COL])
    df.sort_values(TS_COL, inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f'[{label}] {len(df):,} rows | {df[TS_COL].min().date()} to {df[TS_COL].max().date()}')
    return df


def compute_asset_lookbacks(df, last_fail_ts=None, last_rev_ts=None):
    """
    Compute days_since_last_failure and days_since_last_revision
    from inline flags (matching Steiner's _compute_asset_lookbacks).
    Optionally seed with last known timestamps from a previous split.
    Returns (df, last_fail_end_ts, last_rev_end_ts) for chaining.
    """
    fail_col_int = df[FAIL_COL].astype(int)
    maint_col_int = df[MAINT_COL].astype(int)

    # Failure end: transition from failure=1 to failure=0
    fail_end = (fail_col_int == 1) & (fail_col_int.shift(-1) == 0)
    fail_end_ts = pd.Series(pd.NaT, index=df.index)
    fail_end_ts[fail_end] = df.loc[fail_end, TS_COL]
    if last_fail_ts is not None:
        fail_end_ts.iloc[0] = fail_end_ts.iloc[0] if pd.notna(fail_end_ts.iloc[0]) else last_fail_ts
    fail_end_ts = fail_end_ts.ffill()
    df['days_since_last_failure'] = (df[TS_COL] - fail_end_ts).dt.total_seconds() / 86400

    # Revision/maintenance end
    rev_end = (maint_col_int == 1) & (maint_col_int.shift(-1) == 0)
    rev_end_ts = pd.Series(pd.NaT, index=df.index)
    rev_end_ts[rev_end] = df.loc[rev_end, TS_COL]
    if last_rev_ts is not None:
        rev_end_ts.iloc[0] = rev_end_ts.iloc[0] if pd.notna(rev_end_ts.iloc[0]) else last_rev_ts
    rev_end_ts = rev_end_ts.ffill()
    df['days_since_last_revision'] = (df[TS_COL] - rev_end_ts).dt.total_seconds() / 86400

    # Return last known timestamps for chaining to next split
    out_fail = fail_end_ts.iloc[-1] if pd.notna(fail_end_ts.iloc[-1]) else last_fail_ts
    out_rev  = rev_end_ts.iloc[-1]  if pd.notna(rev_end_ts.iloc[-1])  else last_rev_ts
    return df, out_fail, out_rev


def flips(series):
    """Count state transitions (0->1 or 1->0) in a binary series."""
    s = series.values
    if len(s) < 2:
        return 0
    return int(np.sum(np.abs(np.diff((s > 0.5).astype(np.int8)))))


def extract_failure_seconds(df):
    """Return sorted array of failure timestamps as seconds since epoch."""
    fail_ts = df.loc[df[FAIL_COL] == True, TS_COL].drop_duplicates().sort_values()
    return fail_ts.values.astype('datetime64[s]').astype(np.int64)


print('Helpers defined.')

Helpers defined.


## 2. Supervised Dataset (RF Baseline)

Matching Steiner's `create_supervised_feature_datasets.py`:
- **Global epoch-aligned** non-overlapping windows (`ts_s // window_s`)
- Filter out maintenance periods
- Analog: min, max, mean, sum
- Binary: active_s, flips
- Asset features: days_since_last_failure, days_since_last_revision
- Label: shifted failure label (6.2h look-ahead)
- Filter rule: keep windows where `label==1` OR (`label==0` AND `old_label==0`)

In [3]:
def create_supervised_features(df, window_s):
    """
    Create windowed features using epoch-aligned windows
    (matching Steiner's group_by_dynamic / ts_s // window_s approach).
    """
    # Filter out maintenance
    df = df[df[MAINT_COL] != True].copy()
    
    # Compute asset lookbacks BEFORE windowing
    # (Steiner computes these on the full timeline first)
    # Note: already computed on full data before calling this
    
    # Epoch-aligned window IDs
    ts_s = df[TS_COL].values.astype('datetime64[s]').astype(np.int64)
    df['_wid'] = ts_s // window_s
    
    # Aggregations
    agg = {}
    for c in ANALOG_SENSORS:
        if c in df.columns:
            agg[f'{c}_min']  = pd.NamedAgg(column=c, aggfunc='min')
            agg[f'{c}_max']  = pd.NamedAgg(column=c, aggfunc='max')
            agg[f'{c}_mean'] = pd.NamedAgg(column=c, aggfunc='mean')
            agg[f'{c}_sum']  = pd.NamedAgg(column=c, aggfunc='sum')
    for c in BINARY_SENSORS:
        if c in df.columns:
            agg[f'{c}_active_s'] = pd.NamedAgg(column=c, aggfunc='sum')
            agg[f'{c}_flips']    = pd.NamedAgg(column=c, aggfunc=flips)
    
    # Asset lookback: take last value in window
    agg['days_since_last_failure']  = pd.NamedAgg(column='days_since_last_failure', aggfunc='last')
    agg['days_since_last_revision'] = pd.NamedAgg(column='days_since_last_revision', aggfunc='last')
    
    # In-failure flag for current-state label
    agg['_in_failure'] = pd.NamedAgg(column=FAIL_COL, aggfunc='max')
    
    print(f'  Grouping into {window_s}s windows...')
    win = df.groupby('_wid').agg(**agg)
    
    # Convert window ID back to timestamp (window_end = (wid+1) * window_s)
    win.index = pd.to_datetime((win.index + 1) * window_s, unit='s')
    win.index.name = 'window_end'
    
    return win


def create_shifted_labels(window_end_index, failure_ts_s, horizon_s):
    """
    For each window_end T: label=1 if any failure timestamp in [T, T + horizon_s).
    Matching Steiner's shifted label logic.
    """
    if len(failure_ts_s) == 0:
        return np.zeros(len(window_end_index), dtype=np.int8)
    t_arr = window_end_index.values.astype('datetime64[s]').astype(np.int64)
    idx = np.searchsorted(failure_ts_s, t_arr, side='left')
    labels = np.zeros(len(t_arr), dtype=np.int8)
    valid = idx < len(failure_ts_s)
    within = np.zeros(len(t_arr), dtype=bool)
    within[valid] = failure_ts_s[idx[valid]] < t_arr[valid] + horizon_s
    labels[within] = 1
    return labels


def extract_failure_seconds(df):
    """Return sorted array of failure timestamps as seconds since epoch."""
    fail_ts = df.loc[df[FAIL_COL] == True, TS_COL].drop_duplicates().sort_values()
    return fail_ts.values.astype('datetime64[s]').astype(np.int64)


print('Supervised functions defined.')

Supervised functions defined.


In [4]:
# ── Process TRAIN: load, compute lookbacks & features, free ──
print('=== TRAIN ===')
train_raw = load_split(TRAIN_GLOB, 'TRAIN')
train_fail_ts = extract_failure_seconds(train_raw)

print('Computing asset lookbacks...')
train_raw, last_fail, last_rev = compute_asset_lookbacks(train_raw)

print('Computing supervised features...')
feats_train = create_supervised_features(train_raw, SUP_WINDOW_S)
print(f'  -> {len(feats_train):,} windows')

del train_raw; gc.collect()
print('Train raw data freed.\n')

# ── Process TEST: load, seed lookbacks from train, compute features, free ──
print('=== TEST ===')
test_raw = load_split(TEST_GLOB, 'TEST')
test_fail_ts = extract_failure_seconds(test_raw)

print('Computing asset lookbacks (seeded from train)...')
test_raw, _, _ = compute_asset_lookbacks(test_raw, last_fail_ts=last_fail, last_rev_ts=last_rev)

print('Computing supervised features...')
feats_test = create_supervised_features(test_raw, SUP_WINDOW_S)
print(f'  -> {len(feats_test):,} windows')

del test_raw; gc.collect()
print('Test raw data freed.')
print(f'\nFailure seconds: train={len(train_fail_ts):,}, test={len(test_fail_ts):,}')

=== TRAIN ===
[TRAIN] Loading 246 files...


[TRAIN] 14,342,361 rows | 2024-06-01 to 2025-02-11
Computing asset lookbacks...


Computing supervised features...


  Grouping into 350s windows...


  -> 41,375 windows


...

[TEST] 7,901,921 rows | 2025-02-12 to 2025-06-30
Computing asset lookbacks (seeded from train)...


Computing supervised features...


  Grouping into 350s windows...


  -> 23,147 windows


Test raw data freed.

Failure seconds: train=137,489, test=113,467


In [5]:
# ── Compute shifted labels and apply filter rule ──

# Shifted labels (6.2h look-ahead)
feats_train['label'] = create_shifted_labels(feats_train.index, train_fail_ts, SUP_HORIZON_S)
feats_test['label']  = create_shifted_labels(feats_test.index,  test_fail_ts,  SUP_HORIZON_S)

print(f'Windows: train={len(feats_train):,}, test={len(feats_test):,}')

# Filter rule: keep label==1 OR (label==0 AND _in_failure==0)
# Removes ambiguous windows (currently in failure but no future failure in horizon)
mask_train = (feats_train['label'] == 1) | ((feats_train['label'] == 0) & (feats_train['_in_failure'] == 0))
mask_test  = (feats_test['label']  == 1) | ((feats_test['label']  == 0) & (feats_test['_in_failure']  == 0))

sup_train = feats_train[mask_train].drop(columns=['_in_failure']).reset_index()
sup_test  = feats_test[mask_test].drop(columns=['_in_failure']).reset_index()

print(f'\nAfter filter rule:')
print(f'  Train: {len(sup_train):,} windows | label=1: {sup_train["label"].sum()} ({100*sup_train["label"].mean():.2f}%)')
print(f'  Test:  {len(sup_test):,} windows | label=1: {sup_test["label"].sum()} ({100*sup_test["label"].mean():.2f}%)')
print(f'  Feature columns: {len([c for c in sup_train.columns if c not in ["window_end", "label"]])}')

del feats_train, feats_test; gc.collect()

Windows: train=41,375, test=23,147



After filter rule:
  Train: 41,364 windows | label=1: 530 (1.28%)
  Test:  23,134 windows | label=1: 667 (2.88%)
  Feature columns: 340


0

In [6]:
# ── Save supervised datasets ──
sup_train.to_parquet(os.path.join(SUP_OUT, 'train.parquet'), index=False, compression='zstd')
sup_test.to_parquet(os.path.join(SUP_OUT, 'test.parquet'),   index=False, compression='zstd')

print(f'Saved to {SUP_OUT}/')
print(f'  train.parquet: {len(sup_train):,} rows')
print(f'  test.parquet:  {len(sup_test):,} rows')
print(f'\nSample columns: {list(sup_train.columns[:6])} ... {list(sup_train.columns[-4:])}')
print(f'\nTrain label distribution:')
print(sup_train['label'].value_counts().to_string())
print(f'\nTest label distribution:')
print(sup_test['label'].value_counts().to_string())

del sup_train, sup_test; gc.collect()

Saved to C:\Users\Emırhan\Desktop\MASTER THESIS\Kurtulus-thesis\outputs\supervised/
  train.parquet: 41,364 rows
  test.parquet:  23,134 rows

Sample columns: ['window_end', 'CW1_MAIN_RESERVOIR_PRESSURE_min', 'CW1_MAIN_RESERVOIR_PRESSURE_max', 'CW1_MAIN_RESERVOIR_PRESSURE_mean', 'CW1_MAIN_RESERVOIR_PRESSURE_sum', 'CW2_MAIN_RESERVOIR_PRESSURE_min'] ... ['TRAIN_BRAKE_SIGNAL_flips', 'days_since_last_failure', 'days_since_last_revision', 'label']

Train label distribution:
label
0    40834
1      530

Test label distribution:
label
0    22467
1      667


0

## 3. Unsupervised Dataset (LSTM-AE Baseline)

Matching Steiner's `create_unsupervised_sequence_datasets.py`:
- **Non-overlapping** raw sequences of 1402s (23 min)
- **Z-score standardized** using train-normal statistics only
- Filter out maintenance periods
- Splits: train (normal, first 80% of dev), val (normal, last 20% of dev),
  dev_eval (all labels, last 20% of dev), test (all)
- Format: `.npy` arrays of shape `(n_windows, seq_len, n_channels)`

In [7]:
def create_sequences(df, seq_len, channels, min_coverage=0.8):
    """
    Cut raw 1Hz data into non-overlapping windows using epoch-aligned IDs.
    Returns: sequences array (n_windows, seq_len, n_channels),
             window_end timestamps, current failure labels.
    """
    ts_s = df[TS_COL].values.astype('datetime64[s]').astype(np.int64)
    df = df.copy()
    df['_wid'] = ts_s // seq_len
    
    grouped = df.groupby('_wid')
    min_rows = int(seq_len * min_coverage)
    
    sequences = []
    window_ends = []
    labels = []
    
    for wid, group in grouped:
        if len(group) < min_rows:
            continue
        
        # Extract sensor values
        vals = group[channels].values.astype(np.float32)
        
        # Interpolate/pad to exact seq_len if needed
        if len(vals) < seq_len:
            # Linear interpolation to fill to seq_len
            x_old = np.linspace(0, 1, len(vals))
            x_new = np.linspace(0, 1, seq_len)
            padded = np.zeros((seq_len, len(channels)), dtype=np.float32)
            for ch_i in range(len(channels)):
                padded[:, ch_i] = np.interp(x_new, x_old, vals[:, ch_i])
            vals = padded
        elif len(vals) > seq_len:
            vals = vals[:seq_len]
        
        sequences.append(vals)
        window_ends.append(pd.Timestamp((wid + 1) * seq_len, unit='s'))
        labels.append(int(group[FAIL_COL].max()))
    
    if not sequences:
        return np.array([]), np.array([]), np.array([])
    
    return (np.stack(sequences),
            np.array(window_ends),
            np.array(labels, dtype=np.int8))


print('Sequence functions defined.')

Sequence functions defined.


In [8]:
# ── Process TRAIN split ──
print('Loading TRAIN for unsupervised sequences...')
train_raw = load_split(TRAIN_GLOB, 'TRAIN')

# Filter maintenance and select only needed columns to save memory
keep_cols = [TS_COL, FAIL_COL] + [c for c in ALL_SENSORS if c in train_raw.columns]
train_clean = train_raw.loc[train_raw[MAINT_COL] != True, keep_cols]
del train_raw; gc.collect()

# Fill NaN in sensor columns
for c in ALL_SENSORS:
    if c in train_clean.columns:
        train_clean[c] = train_clean[c].fillna(0)

print(f'After maintenance filter: {len(train_clean):,} rows')
print('Creating sequences...')
train_seqs, train_wends, train_labels = create_sequences(
    train_clean, UNSUP_SEQ_S, ALL_SENSORS
)
print(f'Train sequences: {train_seqs.shape}')
print(f'  Normal: {(train_labels == 0).sum()}, Failure: {(train_labels == 1).sum()}')

del train_clean; gc.collect()

Loading TRAIN for unsupervised sequences...
[TRAIN] Loading 246 files...


[TRAIN] 14,342,361 rows | 2024-06-01 to 2025-02-11


After maintenance filter: 13,693,070 rows
Creating sequences...


Train sequences: (9872, 1402, 95)
  Normal: 9807, Failure: 65


0

In [9]:
# ── Process TEST split ──
print('Loading TEST for unsupervised sequences...')
test_raw = load_split(TEST_GLOB, 'TEST')

keep_cols = [TS_COL, FAIL_COL] + [c for c in ALL_SENSORS if c in test_raw.columns]
test_clean = test_raw.loc[test_raw[MAINT_COL] != True, keep_cols]
del test_raw; gc.collect()

for c in ALL_SENSORS:
    if c in test_clean.columns:
        test_clean[c] = test_clean[c].fillna(0)

print(f'After maintenance filter: {len(test_clean):,} rows')
print('Creating sequences...')
test_seqs, test_wends, test_labels = create_sequences(
    test_clean, UNSUP_SEQ_S, ALL_SENSORS
)
print(f'Test sequences: {test_seqs.shape}')
print(f'  Normal: {(test_labels == 0).sum()}, Failure: {(test_labels == 1).sum()}')

del test_clean; gc.collect()

Loading TEST for unsupervised sequences...
[TEST] Loading 136 files...


[TEST] 7,901,921 rows | 2025-02-12 to 2025-06-30


After maintenance filter: 7,611,620 rows
Creating sequences...


Test sequences: (5374, 1402, 95)
  Normal: 5328, Failure: 46


0

In [10]:
# ── Standardize using train-normal statistics ──
normal_mask = train_labels == 0
normal_seqs = train_seqs[normal_mask]  # (n_normal, seq_len, n_channels)

# Compute per-channel mean and std over all normal sequences
# Reshape to (n_normal * seq_len, n_channels) for stats
flat_normal = normal_seqs.reshape(-1, normal_seqs.shape[-1])
ch_mean = flat_normal.mean(axis=0)
ch_std  = flat_normal.std(axis=0)
ch_std[ch_std == 0] = 1.0  # avoid division by zero
del flat_normal

print(f'Channel stats computed from {len(normal_seqs)} normal sequences')
print(f'  Mean range: [{ch_mean.min():.4f}, {ch_mean.max():.4f}]')
print(f'  Std range:  [{ch_std.min():.4f}, {ch_std.max():.4f}]')

# Z-score standardize all sequences
train_seqs = (train_seqs - ch_mean) / ch_std
test_seqs  = (test_seqs  - ch_mean) / ch_std

print('Standardization applied.')

Channel stats computed from 9807 normal sequences
  Mean range: [0.0340, 0.9726]
  Std range:  [0.0284, 0.4975]


Standardization applied.


In [11]:
# ── Split train into train/val/dev_eval (matching Steiner's 80/20 split) ──

# Steiner's split:
#   dev_eval = last 20% of dev (all labels) — for threshold tuning
#   train_pool = first 80% of dev, normal-only → split 80/20 into train/val

n_dev = len(train_seqs)
eval_cutoff = int(n_dev * 0.8)

# dev_eval: last 20% of dev (all labels)
dev_eval_seqs   = train_seqs[eval_cutoff:]
dev_eval_labels = train_labels[eval_cutoff:]
dev_eval_wends  = train_wends[eval_cutoff:]

# train pool: first 80% of dev
pool_seqs   = train_seqs[:eval_cutoff]
pool_labels = train_labels[:eval_cutoff]
pool_wends  = train_wends[:eval_cutoff]

# Normal-only for autoencoder training
pool_normal_mask = pool_labels == 0
normal_pool_seqs   = pool_seqs[pool_normal_mask]
normal_pool_wends  = pool_wends[pool_normal_mask]

# Split normal-only 80/20 into train/val
n_normal = len(normal_pool_seqs)
train_cut = int(n_normal * 0.8)

ae_train_seqs = normal_pool_seqs[:train_cut]
ae_val_seqs   = normal_pool_seqs[train_cut:]

print(f'Unsupervised splits:')
print(f'  train (normal):    {ae_train_seqs.shape}')
print(f'  val (normal):      {ae_val_seqs.shape}')
print(f'  dev_eval (all):    {dev_eval_seqs.shape} | failure: {dev_eval_labels.sum()}')
print(f'  test (all):        {test_seqs.shape} | failure: {test_labels.sum()}')

Unsupervised splits:
  train (normal):    (6272, 1402, 95)
  val (normal):      (1568, 1402, 95)
  dev_eval (all):    (1975, 1402, 95) | failure: 8
  test (all):        (5374, 1402, 95) | failure: 46


In [12]:
# ── Save unsupervised datasets ──

np.save(os.path.join(UNSUP_OUT, 'train.npy'), ae_train_seqs.astype(np.float32))
np.save(os.path.join(UNSUP_OUT, 'val.npy'),   ae_val_seqs.astype(np.float32))
np.save(os.path.join(UNSUP_OUT, 'dev_eval.npy'), dev_eval_seqs.astype(np.float32))
np.save(os.path.join(UNSUP_OUT, 'test.npy'),  test_seqs.astype(np.float32))

# Save labels for dev_eval and test
np.save(os.path.join(UNSUP_OUT, 'dev_eval_labels.npy'), dev_eval_labels)
np.save(os.path.join(UNSUP_OUT, 'test_labels.npy'),     test_labels)

# Save window_end timestamps
pd.Series(dev_eval_wends).to_csv(os.path.join(UNSUP_OUT, 'dev_eval_timestamps.csv'), index=False)
pd.Series(test_wends).to_csv(os.path.join(UNSUP_OUT, 'test_timestamps.csv'), index=False)

# Metadata
stats_dict = {ch: {'mean': float(ch_mean[i]), 'std': float(ch_std[i])}
              for i, ch in enumerate(ALL_SENSORS)}
metadata = {
    'channels': ALL_SENSORS,
    'n_channels': len(ALL_SENSORS),
    'seq_len_s': UNSUP_SEQ_S,
    'label_horizon_s': UNSUP_HORIZON_S,
    'standardization': stats_dict,
    'splits': {
        'train': {'shape': list(ae_train_seqs.shape), 'type': 'normal-only'},
        'val':   {'shape': list(ae_val_seqs.shape),   'type': 'normal-only'},
        'dev_eval': {'shape': list(dev_eval_seqs.shape), 'n_failure': int(dev_eval_labels.sum())},
        'test':  {'shape': list(test_seqs.shape), 'n_failure': int(test_labels.sum())},
    }
}
with open(os.path.join(UNSUP_OUT, 'metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Saved to {UNSUP_OUT}/')
for fname in ['train.npy', 'val.npy', 'dev_eval.npy', 'test.npy', 'metadata.json']:
    fpath = os.path.join(UNSUP_OUT, fname)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f'  {fname}: {size_mb:.1f} MB')

Saved to C:\Users\Emırhan\Desktop\MASTER THESIS\Kurtulus-thesis\outputs\unsupervised/
  train.npy: 3341.5 MB
  val.npy: 835.4 MB
  dev_eval.npy: 1052.2 MB
  test.npy: 2863.1 MB
  metadata.json: 0.0 MB


## Summary

Datasets created matching Steiner's baseline pipeline:

**Supervised (RF baseline):** `outputs/supervised/`
- Window: 350s (6 min), Label horizon: 22326s (6.2h)
- Features: analog (min/max/mean/sum) + binary (active_s/flips) + asset lookback
- Steiner baseline: dev F1 = 0.98, test F1 = 0.21

**Unsupervised (LSTM-AE baseline):** `outputs/unsupervised/`
- Sequence: 1402s (23 min), Label horizon: 11163s (3.1h)
- Z-score standardized using train-normal statistics
- Steiner baseline: dev F1 = 0.75, test F1 = 0.08

**Next step (Phase 1 / RQ1):** Anomaly analysis and clustering to construct
subsystem-aware feature representations from these datasets.